# 27b. Agent Evals

**Tier:** Evaluation & Production
**Estimated time:** 60 minutes
**Prerequisites:** 19, 23, 24, 26
**Priority:** 🔴 Crucial — agent evals (trajectories, tool calls, cost-per-solved-task) are the rarest skill on the market right now; model evals alone don't transfer to agent systems, and this is the notebook that lets you *prove* a harness change helped instead of just believing it did. *If skipped, revisit when:* n/a for anyone building agents — this is the "shine above others" notebook.
**Source material:** @sairahul1 Harness Engineering (the +36-point claim) — https://x.com/sairahul1/status/2063544956158185927 ; Stanford Lecture 8

## What You'll Learn
- Trajectory scoring: judging *how* an agent got to an answer, not just whether the answer is right
- Tool-call correctness: did it call the tools it needed, avoid the ones it shouldn't, with valid arguments?
- Building a mini task suite (a tiny τ-bench-style harness) and running an agent through it
- Cost-per-solved-task as the metric that ties quality and efficiency together
- Using this harness to actually measure the harness-engineering claim from notebook 21: does a better harness measurably improve the score?

## Why This Matters
Notebook 24 built an eval harness for single-turn Q&A. Agents are different: the same question can be answered via many different tool-call paths, some efficient and correct, some wasteful or subtly wrong despite reaching the right final answer. If you can't score a *trajectory*, you can't tell a genuinely better agent harness from one that just got lucky on a fixed test set — and you can't prove notebook 21's "+36 points, same model" claim without exactly this kind of measurement.


In [ ]:
import os, json, time
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — live agent runs will be skipped.")


## A minimal tool-using agent (self-contained, in the spirit of notebook 19)

To evaluate an agent, we need one to evaluate. This is a small raw-loop agent (like notebook 19) with two tools over a mock knowledge base, deliberately kept simple so the eval logic — not the agent — is the point of this notebook.

In [ ]:
_KB = {
    "python": "Python is a high-level, dynamically-typed programming language created by Guido van Rossum in 1991.",
    "rust": "Rust is a systems programming language focused on memory safety without a garbage collector, first released in 2010.",
    "attention": "The attention mechanism lets a model weigh how relevant each input token is to every other token (see notebook 4).",
    "kv cache": "KV caching stores past attention keys/values so autoregressive decoding doesn't recompute them each step (see notebook 13).",
}

def search_docs(query):
    q = query.lower()
    hits = [v for k, v in _KB.items() if k in q]
    return " ".join(hits) if hits else "No results found."

def calculator(expression):
    # Restricted eval: digits and arithmetic operators only.
    import re
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: invalid characters in expression."
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

TOOLS = {
    "search_docs": search_docs,
    "calculator": calculator,
}
TOOL_SCHEMAS = [
    {"name": "search_docs", "description": "Search internal docs for a topic.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "calculator", "description": "Evaluate a basic arithmetic expression.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
]

def run_agent(task_prompt, system_prompt="Answer the user's question. Use tools when helpful, then give a final answer.", max_steps=5):
    """Raw tool loop (notebook 19 shape). Returns a trajectory: every tool call + the final answer."""
    if not HAS_ANTHROPIC:
        return {"trajectory": [], "final_answer": "[skipped: no ANTHROPIC_API_KEY]", "cost_tokens": 0}

    messages = [{"role": "user", "content": task_prompt}]
    trajectory = []
    total_tokens = 0
    for step in range(max_steps):
        resp = client.messages.create(model=TEACH_MODEL, max_tokens=300, system=system_prompt,
                                       tools=TOOL_SCHEMAS, messages=messages)
        total_tokens += resp.usage.input_tokens + resp.usage.output_tokens
        if resp.stop_reason != "tool_use":
            final_text = " ".join(b.text for b in resp.content if b.type == "text")
            return {"trajectory": trajectory, "final_answer": final_text, "cost_tokens": total_tokens}
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                output = TOOLS[b.name](**b.input)
                trajectory.append({"tool": b.name, "input": b.input, "output": output})
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": output})
        messages.append({"role": "user", "content": results})
    return {"trajectory": trajectory, "final_answer": "[budget exhausted]", "cost_tokens": total_tokens}

demo = run_agent("What year was Python created? Use search_docs.")
print(json.dumps(demo, indent=2)[:500])


## Trajectory scoring — were the steps reasonable, not just the answer?

Two agents can reach the same correct final answer through very different paths: one calls the right tool once, another flails through five irrelevant calls before stumbling onto the answer. A trajectory scorer grades the *path*, using a rubric similar to notebook 24's rubric scorer but over the tool-call sequence instead of the text output.

In [ ]:
def trajectory_score(trajectory, required_tools, max_reasonable_steps):
    """Rubric over the path taken, independent of whether the final answer was correct."""
    used_tools = {step["tool"] for step in trajectory}
    used_required = required_tools.issubset(used_tools)
    within_budget = len(trajectory) <= max_reasonable_steps
    no_repeated_calls = len(trajectory) == len({(s["tool"], json.dumps(s["input"], sort_keys=True)) for s in trajectory})
    return {
        "used_required_tools": used_required,
        "within_budget": within_budget,
        "no_wasted_repeats": no_repeated_calls,
        "score": np.mean([used_required, within_budget, no_repeated_calls]),
    }

score = trajectory_score(demo["trajectory"], required_tools={"search_docs"}, max_reasonable_steps=2)
print(json.dumps(score, indent=2))


## Tool-call correctness — did it call the RIGHT tools, correctly?

Beyond "was the path reasonable," tool-call correctness checks each call individually: valid arguments, no calls to tools that shouldn't have been needed, and no malformed inputs that a real API would reject. This is the agent-specific analogue of notebook 24's exact-match scorer — deterministic, cheap, and it catches a different failure class than trajectory scoring does.

In [ ]:
def tool_call_correctness(trajectory, forbidden_tools=frozenset()):
    problems = []
    for i, step in enumerate(trajectory):
        if step["tool"] in forbidden_tools:
            problems.append(f"step {i}: called forbidden tool {step['tool']!r}")
        if isinstance(step["output"], str) and step["output"].startswith("Error:"):
            problems.append(f"step {i}: tool call errored — {step['output']}")
    return {"n_problems": len(problems), "problems": problems, "clean": len(problems) == 0}

correctness = tool_call_correctness(demo["trajectory"], forbidden_tools={"calculator"})
print(json.dumps(correctness, indent=2))


## A mini task suite (τ-bench-style)

A real agent-eval suite (τ-bench and similar) is a list of tasks, each with: a prompt, the tools that SHOULD be used, a budget, and a way to check the final answer. We build a tiny one here — small enough to run in seconds, but structurally identical to a production suite.

In [ ]:
TASK_SUITE = [
    {
        "id": "t1", "prompt": "What year was Python created? Use search_docs to check, then answer.",
        "required_tools": {"search_docs"}, "forbidden_tools": {"calculator"},
        "answer_check": lambda ans: "1991" in ans,
    },
    {
        "id": "t2", "prompt": "What is 47 * 12? Use the calculator tool.",
        "required_tools": {"calculator"}, "forbidden_tools": {"search_docs"},
        "answer_check": lambda ans: "564" in ans,
    },
    {
        "id": "t3", "prompt": "What does KV caching do? Use search_docs, then summarize in one sentence.",
        "required_tools": {"search_docs"}, "forbidden_tools": {"calculator"},
        "answer_check": lambda ans: "cach" in ans.lower(),
    },
]

def evaluate_task(task, system_prompt=None):
    kwargs = {"max_steps": 5}
    if system_prompt:
        kwargs["system_prompt"] = system_prompt
    result = run_agent(task["prompt"], **kwargs)
    traj_score = trajectory_score(result["trajectory"], task["required_tools"], max_reasonable_steps=2)
    correctness = tool_call_correctness(result["trajectory"], task["forbidden_tools"])
    solved = task["answer_check"](result["final_answer"]) if HAS_ANTHROPIC else False
    return {
        "id": task["id"], "solved": solved, "trajectory_score": traj_score["score"],
        "correctness_clean": correctness["clean"], "cost_tokens": result["cost_tokens"],
    }

results = [evaluate_task(t) for t in TASK_SUITE]
for r in results:
    print(r)


## Cost-per-solved-task

Quality and efficiency are both real constraints — an agent that solves every task by calling every tool ten times "works" but is unaffordable at scale. Cost-per-solved-task ties the two together into one number you can compare across agent/harness versions, directly answering "is this new version actually better, all-in?" 

In [ ]:
def cost_per_solved_task(results):
    solved = [r for r in results if r["solved"]]
    if not solved:
        return None
    total_cost_tokens = sum(r["cost_tokens"] for r in results)   # charge for ALL attempts, not just solved ones
    return total_cost_tokens / len(solved)

cpst = cost_per_solved_task(results)
solve_rate = np.mean([r["solved"] for r in results]) if HAS_ANTHROPIC else 0.0
print(f"Solve rate: {solve_rate:.0%}")
print(f"Cost per solved task: {cpst:.0f} tokens" if cpst else "Cost per solved task: n/a (nothing solved)")


## Eval-driven harness iteration — measuring the +36-point claim

Notebook 21 claimed a better harness can move the same model up to +36 points on a benchmark. Here's how you'd actually check that claim instead of taking it on faith: run the SAME task suite through two system prompts — a bare one and a harnessed one that gives the agent explicit tool-use guidance — and compare the aggregate trajectory + solve scores.

In [ ]:
BARE_SYSTEM = "Answer the user's question."
HARNESSED_SYSTEM = (
    "Answer the user's question. Before answering, decide if search_docs or calculator would make "
    "your answer more reliable, and call it first if so. Never guess a fact you could verify. "
    "Give a short, direct final answer once you have enough information."
)

bare_results = [evaluate_task(t, system_prompt=BARE_SYSTEM) for t in TASK_SUITE]
harnessed_results = [evaluate_task(t, system_prompt=HARNESSED_SYSTEM) for t in TASK_SUITE]

def summarize(results, label):
    solve_rate = np.mean([r["solved"] for r in results]) if HAS_ANTHROPIC else 0.0
    traj = np.mean([r["trajectory_score"] for r in results])
    print(f"{label:12s} solve_rate={solve_rate:.0%}  avg_trajectory_score={traj:.2f}")

summarize(bare_results, "bare")
summarize(harnessed_results, "harnessed")


## Exercises

**Exercise 1 (Warm-up):** Add a fourth task to `TASK_SUITE` (your choice of tool/question) and confirm `evaluate_task` scores it correctly for both a good and a deliberately bad `answer_check`.

**Exercise 2 (Apply):** Implement `trajectory_score`'s missing failure mode — a check for **tool-order sensitivity**: for tasks where `search_docs` must happen before `calculator` (e.g. "look up a rate, then compute a total"), penalize trajectories that call them out of order.

**Exercise 3 (Extend):** This suite has 3 tasks. Sketch (in comments) how you'd scale this into a real regression gate for notebook 33's CI pipeline — what changes about task count, thresholds, and how failures should block a merge versus just warn?


In [ ]:
# Exercise 1: Warm-up
# Task: Add a 4th task to TASK_SUITE and evaluate it.
# Hint: reuse the dict shape of existing tasks; answer_check is just a function on the final string.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Add order-sensitivity checking to trajectory_score for tasks that declare an expected order.
# Hint: compare the index of the first "search_docs" call to the first "calculator" call.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch scaling this suite into a CI regression gate (notebook 33).
# Hint: consider a MUCH larger frozen task suite, a required solve-rate floor, and treating a
# trajectory-score regression as a warning while a solve-rate regression blocks the merge.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
TASK_SUITE.append({
    "id": "t4", "prompt": "What is Rust known for? Use search_docs.",
    "required_tools": {"search_docs"}, "forbidden_tools": {"calculator"},
    "answer_check": lambda ans: "memory" in ans.lower() or "safety" in ans.lower(),
})
print(evaluate_task(TASK_SUITE[-1]))

# Exercise 2
def trajectory_score_ordered(trajectory, required_tools, max_reasonable_steps, expected_order=None):
    base = trajectory_score(trajectory, required_tools, max_reasonable_steps)
    if expected_order:
        tool_sequence = [step["tool"] for step in trajectory]
        indices = [tool_sequence.index(t) for t in expected_order if t in tool_sequence]
        in_order = indices == sorted(indices)
        base["correct_order"] = in_order
        base["score"] = np.mean([base["used_required_tools"], base["within_budget"],
                                  base["no_wasted_repeats"], in_order])
    return base

# Exercise 3
# A CI-grade suite would need: (1) 30-100+ frozen tasks covering the agent's real tool surface,
# never edited by the same PR that changes the harness; (2) a required solve-rate floor (e.g.
# "must stay >= 90% of the last green baseline") that FAILS the build if breached; (3) trajectory
# and cost-per-solved-task regressions logged as a WARNING comment on the PR rather than a hard
# block, since efficiency regressions are worth flagging but rarely worth blocking a ship on their
# own — exactly the "gate on correctness, warn on quality" split notebook 33 implements.
```
</details>

## Key Takeaways
- Trajectory scoring judges *how* an agent reached an answer (tool choice, step budget, no wasted repeats) — a different failure class than whether the final answer was right.
- Tool-call correctness is the agent-specific analogue of exact-match scoring: deterministic, cheap, and catches malformed or forbidden calls.
- A mini task suite (prompt + required/forbidden tools + answer check) is the same shape as production agent-eval frameworks like τ-bench, just smaller.
- Cost-per-solved-task ties quality and efficiency into one number — the metric that actually answers "is this agent version better, all-in?"
- You can now measure claims like "a better harness adds +36 points" directly, instead of trusting them — run the same task suite through both system prompts and compare solve rate and trajectory score.

## What's Next
Notebook 27c wires these scores into control flow -- a verdict that doesn't change what runs next is a report, not a gate. Then Tier 8 (notebook 28 onward) shifts from measuring agents to hardening them for production: structured outputs, serving, security, and cost engineering.
